# Dual-Core Agent: Selkov Convergence &rarr; Goldilocks ZK Proof

A two-core hardware pipeline simulated with **cocotb** + **Icarus Verilog**:

1. **Core 1 &mdash; `selkov` (the Executor).** A fixed-point (Q8.24) Selkov
   glycolytic oscillator integrated with forward Euler. With `a = 0.5`,
   `b = 0.7` the Jacobian trace at the fixed point is `-1.0`, so the state
   **converges** to `x* = b = 0.7`, `y* = b/(a+b^2) = 0.707`. All arithmetic
   saturates and raises a sticky fault flag &mdash; silent wraparound is a
   fault, not a mode.

2. **Core 2 &mdash; `zk_goldilocks_r1cs` (the Prover).** Once the executor has
   settled, one trigger pulse computes the R1CS witness
   `C = state^2 mod PRIME` over the Goldilocks prime
   `PRIME = 2^64 - 2^32 + 1`, and we check it bit-exactly against Python.

**Requirements:** `iverilog`, and `pip install cocotb pandas matplotlib`.
Run the cells top to bottom.

## 1. Hardware &mdash; `agent_core.sv`

In [ ]:
sv_code = r'''`timescale 1ns/1ps
`default_nettype none          // implicit wires become compile errors, not silent bugs

//=====================================================================
// agent_core.sv - Dual-core "convergence + ZK" agent
//
//   CORE 1 (selkov)          : fixed-point Selkov executor that
//                              CONVERGES to a fixed point, then
//   CORE 2 (zk_goldilocks_r1cs): proves the settled state with an
//                              R1CS witness over the Goldilocks field.
//=====================================================================

//---------------------------------------------------------------------
// CORE 1 : fixed-point Selkov glycolytic oscillator (the Executor)
//
// Implements exactly:
//     dx/dt = -x + a*y + x^2*y
//     dy/dt =  b - a*y - x^2*y
//
// Format:      Q8.24 signed (8 integer incl. sign, 24 fractional)
// Integration: forward Euler,  v_{n+1} = v_n + (dv/dt)*dt
//
// Limit cycle requires trace(J) > 0 at the fixed point x*=b, y*=b/(a+b^2):
//     trace = -1 + 2*b*y* - a - b^2
//   a=0.08, b=0.60 -> trace = +0.196  -> limit cycle
//   a=0.50, b=0.70 -> trace = -1.000  -> decays to a fixed point (convergence)
//
// Arithmetic saturates and raises a sticky fault flag. Silent
// wraparound is treated as a fault, not a mode.
//
// Helper functions return 33 bits: [32] = overflow, [31:0] = value.
//---------------------------------------------------------------------
module selkov (
    input  wire               clk,
    input  wire               rst,
    input  wire signed [31:0] a,        // Q8.24
    input  wire signed [31:0] b,        // Q8.24
    input  wire signed [31:0] dt,       // Q8.24
    output reg  signed [31:0] x,        // Q8.24
    output reg  signed [31:0] y,        // Q8.24
    output reg                saturated // sticky fault
);
    localparam signed [31:0] SAT_POS = 32'sh7FFFFFFF;
    localparam signed [31:0] SAT_NEG = 32'sh80000000;

    // Q8.24 * Q8.24 -> Q16.48; the Q8.24 slice is bits [55:24].
    // Valid only if [63:55] is a proper sign extension of bit 55.
    function [32:0] qmul (input signed [31:0] A, input signed [31:0] B);
        reg signed [63:0] t;
        begin
            t = A * B;
            if (t[63:55] != {9{t[55]}})
                qmul = {1'b1, (t[63] ? SAT_NEG : SAT_POS)};
            else
                qmul = {1'b0, t[55:24]};
        end
    endfunction

    function [32:0] qadd (input signed [31:0] A, input signed [31:0] B);
        reg signed [32:0] t;
        begin
            t = {A[31], A} + {B[31], B};
            if (t[32] != t[31])
                qadd = {1'b1, (t[32] ? SAT_NEG : SAT_POS)};
            else
                qadd = {1'b0, t[31:0]};
        end
    endfunction

    wire [32:0] r_xsq   = qmul(x, x);                        // x^2
    wire [32:0] r_xsqy  = qmul(r_xsq[31:0], y);              // x^2*y
    wire [32:0] r_ay    = qmul(a, y);                        // a*y

    wire [32:0] r_t1    = qadd(-x, r_ay[31:0]);              // -x + a*y
    wire [32:0] r_dxdt  = qadd(r_t1[31:0], r_xsqy[31:0]);    // + x^2*y
    wire [32:0] r_t2    = qadd(b, -r_ay[31:0]);              //  b - a*y
    wire [32:0] r_dydt  = qadd(r_t2[31:0], -r_xsqy[31:0]);   // - x^2*y

    wire [32:0] r_dx    = qmul(r_dxdt[31:0], dt);
    wire [32:0] r_dy    = qmul(r_dydt[31:0], dt);
    wire [32:0] r_xn    = qadd(x, r_dx[31:0]);
    wire [32:0] r_yn    = qadd(y, r_dy[31:0]);

    wire any_ovf = r_xsq[32] | r_xsqy[32] | r_ay[32] | r_t1[32] | r_dxdt[32]
                 | r_t2[32]  | r_dydt[32] | r_dx[32] | r_dy[32]
                 | r_xn[32]  | r_yn[32];

    always @(posedge clk) begin
        if (rst) begin
            x         <= 32'sh00800000;   // 0.5
            y         <= 32'sh00800000;   // 0.5
            saturated <= 1'b0;
        end else begin
            x <= r_xn[31:0];
            y <= r_yn[31:0];
            if (any_ovf) saturated <= 1'b1;
        end
    end
endmodule

//---------------------------------------------------------------------
// CORE 2 : Goldilocks-field ZK R1CS prover
//
// On a trigger, computes the witness for the R1CS constraint
//     C = (state * state) mod PRIME
// over the Goldilocks prime PRIME = 2^64 - 2^32 + 1.
//---------------------------------------------------------------------
module zk_goldilocks_r1cs (
    input  wire        clk,
    input  wire        trigger,
    input  wire [31:0] target_state,
    output reg  [63:0] witness_c,
    output reg         proof_ready
);
    localparam [63:0] PRIME = 64'hFFFFFFFF00000001; // 2^64 - 2^32 + 1

    always @(posedge clk) begin
        if (trigger) begin
            // 32x32 -> 64-bit product always fits below PRIME's 64 bits,
            // so the reduction is a single modulo.
            witness_c   <= ($unsigned(target_state) * $unsigned(target_state)) % PRIME;
            proof_ready <= 1'b1;
        end else begin
            proof_ready <= 1'b0;
        end
    end
endmodule

//---------------------------------------------------------------------
// TOP LEVEL : route the executor's settled x-state into the prover
//---------------------------------------------------------------------
module agent_core (
    input  wire               clk,
    input  wire               rst,
    input  wire               zk_trigger,
    input  wire signed [31:0] a,
    input  wire signed [31:0] b,
    input  wire signed [31:0] dt,
    output wire signed [31:0] x_out,
    output wire signed [31:0] y_out,
    output wire               saturated,
    output wire        [63:0] zk_witness,
    output wire               zk_ready
);
    selkov drift_engine (
        .clk(clk), .rst(rst),
        .a(a), .b(b), .dt(dt),
        .x(x_out), .y(y_out), .saturated(saturated)
    );

    zk_goldilocks_r1cs prover_engine (
        .clk(clk), .trigger(zk_trigger),
        .target_state(x_out),
        .witness_c(zk_witness), .proof_ready(zk_ready)
    );
endmodule

`default_nettype wire
'''
with open('agent_core.sv', 'w') as f:
    f.write(sv_code)
print('wrote agent_core.sv ({} bytes)'.format(len(sv_code)))

## 2. Testbench &mdash; `test_zk_pipeline.py` (cocotb 2.x)

In [ ]:
tb_code = r'''"""cocotb testbench: settle the Selkov executor, then prove its state.

Written for cocotb 2.x (Clock uses `unit=`, failures use `assert`).
"""
import csv

import cocotb
from cocotb.clock import Clock
from cocotb.triggers import RisingEdge

Q = 24  # Q8.24 fractional bits


def float_to_q(val):
    return int(round(val * (1 << Q)))


def q_to_float(val):
    val &= 0xFFFFFFFF
    if val & (1 << 31):
        val -= (1 << 32)
    return val / (1 << Q)


# Goldilocks prime: 2^64 - 2^32 + 1
PRIME = (2 ** 64) - (2 ** 32) + 1

SETTLE_TICKS = 6000   # let the executor converge to its fixed point
POST_TICKS = 400      # cycles logged after the proof, for plotting


@cocotb.test()
async def run_zk_pipeline(dut):
    cocotb.start_soon(Clock(dut.clk, 10, unit="ns").start())

    # a=0.5, b=0.7 -> trace = -1.0 -> converges to a fixed point.
    dut.rst.value = 1
    dut.zk_trigger.value = 0
    dut.a.value = float_to_q(0.5)
    dut.b.value = float_to_q(0.7)
    dut.dt.value = float_to_q(0.01)

    await RisingEdge(dut.clk)
    dut.rst.value = 0
    for _ in range(5):            # let signals propagate out of reset
        await RisingEdge(dut.clk)

    with open("zk_pipeline_data.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["tick", "x", "y", "zk_ready", "zk_witness"])

        # Phase 1: allow the executor to settle.
        for i in range(SETTLE_TICKS):
            await RisingEdge(dut.clk)
            writer.writerow([i,
                             q_to_float(int(dut.x_out.value)),
                             q_to_float(int(dut.y_out.value)),
                             0, 0])

        assert int(dut.saturated.value) == 0, "executor saturated before proof"

        # Capture the exact state the hardware prover will consume.
        final_x_bits = int(dut.x_out.value) & 0xFFFFFFFF

        # Phase 2: fire the ZK R1CS prover for one cycle.
        dut.zk_trigger.value = 1
        await RisingEdge(dut.clk)
        dut.zk_trigger.value = 0

        await RisingEdge(dut.clk)   # witness registered on this edge
        hw_witness = int(dut.zk_witness.value)
        assert int(dut.zk_ready.value) == 1, "proof_ready did not assert"

        python_witness = (final_x_bits * final_x_bits) % PRIME

        dut._log.info("--- ZK R1CS VERIFICATION ---")
        dut._log.info("Final state (x bits) : %d", final_x_bits)
        dut._log.info("Hardware witness     : %d", hw_witness)
        dut._log.info("Python witness       : %d", python_witness)
        assert hw_witness == python_witness, (
            "Goldilocks witness mismatch: hw=%d python=%d"
            % (hw_witness, python_witness))
        dut._log.info("RESULT: EXACT MATCH. Goldilocks field accelerated on silicon.")

        # Phase 3: log a tail with the proof asserted, for plotting.
        for i in range(SETTLE_TICKS + 1, SETTLE_TICKS + 1 + POST_TICKS):
            await RisingEdge(dut.clk)
            writer.writerow([i,
                             q_to_float(int(dut.x_out.value)),
                             q_to_float(int(dut.y_out.value)),
                             1, hw_witness])
'''
with open('test_zk_pipeline.py', 'w') as f:
    f.write(tb_code)
print('wrote test_zk_pipeline.py ({} bytes)'.format(len(tb_code)))

## 3. Makefile

In [ ]:
make_code = r'''TOPLEVEL_LANG   = verilog
VERILOG_SOURCES = agent_core.sv
TOPLEVEL        = agent_core
MODULE          = test_zk_pipeline

# SystemVerilog features (typed functions, `default_nettype none`) need -g2012.
COMPILE_ARGS   += -g2012

include $(shell cocotb-config --makefiles)/Makefile.sim
'''
with open('Makefile', 'w') as f:
    f.write(make_code)
print('wrote Makefile')

## 4. Run the simulation

In [ ]:
import subprocess, os

# Clean any previous run so results are fresh.
for p in ('sim_build', 'results.xml', 'zk_pipeline_data.csv'):
    subprocess.run(['rm', '-rf', p], check=False)

proc = subprocess.run(
    'make SIM=icarus', shell=True,
    capture_output=True, text=True)
print(proc.stdout[-4000:])
if proc.returncode != 0:
    print(proc.stderr[-4000:])
    raise SystemExit('simulation failed (see output above)')

# cocotb records failures in results.xml even when make returns 0.
import xml.etree.ElementTree as ET
root = ET.parse('results.xml').getroot()
failures = sum(len(tc.findall('failure')) + len(tc.findall('error'))
               for tc in root.iter('testcase'))
assert failures == 0, 'cocotb reported {} failing testcase(s)'.format(failures)
print('\nSimulation passed: ZK witness verified bit-exactly.')

## 5. Plot: executor settling and the proof event

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('zk_pipeline_data.csv')
zk_event = df[df['zk_ready'] == 1].iloc[0]
print('ZK witness  :', int(zk_event['zk_witness']))
print('settled x,y :', round(zk_event['x'], 6), round(zk_event['y'], 6))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Time series with the ZK trigger marked.
ax1.plot(df['tick'], df['x'], label='x (info state)', color='blue', alpha=0.8)
ax1.plot(df['tick'], df['y'], label='y (action tendency)', color='orange', alpha=0.8)
ax1.axvline(x=zk_event['tick'], color='red', linestyle='--', linewidth=2,
            label='ZK proof generated')
ax1.set_title('Executor settling -> ZK proof generation')
ax1.set_xlabel('hardware clock ticks')
ax1.set_ylabel('Q8.24 magnitude')
ax1.legend(); ax1.grid(True)

# Phase plane, converging onto the fixed point.
ax2.plot(df['x'], df['y'], color='purple', linewidth=1.0, alpha=0.6)
ax2.scatter(zk_event['x'], zk_event['y'], color='red', s=100, zorder=5,
            label='settled state (proved)')
ax2.set_title('Phase plane: lock-in at the fixed point')
ax2.set_xlabel('x (info state)'); ax2.set_ylabel('y (action tendency)')
ax2.legend(); ax2.grid(True)

plt.tight_layout()
plt.show()